In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
car_df=pd.read_csv("cardekho_dataset.csv",index_col=[0])
car_df.head()

#### perform feature engineering

In [ ]:
car_df_copy=car_df.copy()

In [ ]:
car_df_copy.head()

In [ ]:
# check missing values
car_df_copy.isnull().any(axis=0)

### Apparently, there are no missing values in the dataset

In [ ]:
# check column types
car_df_copy.info()

In [ ]:
car_df_copy["car_name"].value_counts()

In [ ]:
# drop car_name column
car_df_copy.drop(labels="car_name",axis=1,inplace=True)

In [ ]:
car_df_copy.head()

In [ ]:
car_df_copy.info()

In [ ]:
# brand column
car_df_copy["brand"].unique()

In [ ]:
# remove the brand column
car_df_copy.drop(labels="brand",axis=1,inplace=True)
car_df_copy.head()

In [ ]:
car_df_copy.info()

In [ ]:
# model column
car_df_copy["model"].unique()

In [ ]:
# separating categorical and numerical features
num_features=[feature for feature in car_df_copy.columns if car_df_copy[feature].dtype!='O']
cat_features=[feature for feature in car_df_copy.columns if car_df_copy[feature].dtype=='O']

print(num_features)
print(cat_features)

In [ ]:
# separating independent and dependent features
X=car_df_copy.drop(labels="selling_price",axis=1)
y=car_df_copy["selling_price"]

In [ ]:
X.head()

In [ ]:
y

#### Feature encoding and scaling

In [ ]:
# model column
len(X["model"].unique())

##### 'model' has 120 unique values, so performing OHE is not a very good idea, let's apply Label encoding since it will also give us the relationship between 'model' and 'selling_price'

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_encoder=LabelEncoder()
X["model"]=le_encoder.fit_transform(X["model"])

In [ ]:
X.head()

In [ ]:
len(X["seller_type"].unique()),len(X["fuel_type"].unique()),len(X["transmission_type"].unique())

##### So for these columns, I can apply OHE, since very less categories, and rest on the numeric features apply standardscaling

In [ ]:
# before that apply train test split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

In [ ]:
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

ohe_columns=["seller_type","fuel_type","transmission_type"]
num_features=X.select_dtypes(exclude="object").columns

ohe_transformer=OneHotEncoder(drop="first")
scale_transformer=StandardScaler()

column_transform=ColumnTransformer(
    [
    ('ohe_transform',ohe_transformer,ohe_columns),
    ('scale_transformer',scale_transformer,num_features)
],remainder="passthrough")

In [ ]:
# apply fit_transform on training data
X_train=column_transform.fit_transform(X_train)

# apply transform on test data
X_test=column_transform.transform(X_test)

In [ ]:
X_train=pd.DataFrame(X_train,columns=column_transform.get_feature_names_out())
X_test=pd.DataFrame(X_test,columns=column_transform.get_feature_names_out())

# Now we'll do model training and model selection

In [ ]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error

In [ ]:
# a function to evaluate model
def evaluate_model(true_val,predicted_val):
    r2_val=r2_score(y_true=true_val,y_pred=predicted_val)
    mse=mean_squared_error(y_true=true_val,y_pred=predicted_val)
    mae=mean_absolute_error(y_true=true_val,y_pred=predicted_val)
    print(f"The R2 score is: {r2_val}")
    print(f"The mean square error: {mse}")
    print(f"The mean absolute error: {mae}")

In [ ]:
# begin the model training
models={
    "Linear Regression":LinearRegression(n_jobs=-1),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "SVR":SVR(),
    "KNN Regressor":KNeighborsRegressor(n_jobs=-1),
    "Decision Tree Regressor":DecisionTreeRegressor(),
    "Random Forest Regressor":RandomForestRegressor(n_jobs=-1),
    "AdaBoost Regressor":AdaBoostRegressor(),
    "Gradient Boosting Regressor":GradientBoostingRegressor()
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    # train the model
    model.fit(X_train,y_train)

    # get predicted values for train data
    y_train_pred=model.predict(X_train)
    # get predicted values for test data
    y_test_pred=model.predict(X_test)

    print(f"Results for model: {list(models.keys())[i]}")
    # evaluate the model for training accuracy
    evaluate_model(true_val=y_train,predicted_val=y_train_pred)
    # evaluate the model for testing accuracy
    evaluate_model(true_val=y_test,predicted_val=y_test_pred)
    print("==============================================================")

### Perform hyperparameter tuning for Gradient-Boost regressor model

In [ ]:
# hyperparameters for gradient-boost
loss=['squared_error', 'absolute_error', 'huber', 'quantile']
learning_rate=[0.01,0.1,1,10]
subsample=[0.5,0.1,1]
criterion=['friedman_mse','squared_error']
min_samples_split=[3,5,10]
min_samples_leaf=[3,5,10]
max_depth=[5,10]

gboost_params=dict(loss=loss,learning_rate=learning_rate,subsample=subsample,criterion=criterion,min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,max_depth=max_depth)

In [ ]:
# perform hyperparameter tuning
randomcv_models=[
    ("Gradient Boosting Regressor",GradientBoostingRegressor(),gboost_params)
]

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

model_params={}
for name,model,params in randomcv_models:
    random=RandomizedSearchCV(estimator=model,param_distributions=params,n_iter=20,n_jobs=-1,cv=3)
    random.fit(X_train,y_train)

    model_params[name]=random.best_params_

for model_name in model_params:
    print(f"---------------Best params for {model_name}------------------------")
    print(model_params[model_name])

In [ ]:
# now train the models with these params
models={
    "Gradient Boosting Regressor":GradientBoostingRegressor(subsample=1, min_samples_split=3, min_samples_leaf=3, max_depth=10, loss='absolute_error', learning_rate=0.1, criterion='friedman_mse')
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    # train the model
    model.fit(X_train,y_train)

    # predict for train data
    y_train_pred=model.predict(X_train)
    # predict for test data
    y_test_pred=model.predict(X_test)

    model_name=list(models.keys())[i]
    print(f"Evaluation for {model_name}")
    print("------Train Data------")
    evaluate_model(y_train,y_train_pred)
    print("------Test Data------")
    evaluate_model(y_test,y_test_pred)
    print("================================")